In [1]:
!pip install semantic-link-sempy semantic-link-labs
!pip install --upgrade semantic-link --q


StatementMeta(, 780df7fd-f32f-4aba-9a1c-c0e4bccb6235, 3, Finished, Available, Finished, False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.0/888.0 kB 10.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 50.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: azure-core
    Found existing installation: azure-core 1.30.2
    Uninstalling azure-core-1.30.2:
      Successfully uninstalled azure-core-1.30.2
  Attempting uninstall: semantic-link-sempy
    Found existing installation: semantic-link-sempy 0.11.0
    Uninstalling semantic-link-sempy-0.11.0:
      Successfully uninst

In [3]:
!pip install notebookutils

StatementMeta(, 780df7fd-f32f-4aba-9a1c-c0e4bccb6235, 5, Finished, Available, Finished, False)

In [2]:
import os
import re
import json
import base64
#import pos
from datetime import datetime, timezone
import time

import pandas as pd
import sempy.fabric as fabric
import sempy_labs as labs  # optional, but useful to keep available

import notebookutils
import requests

StatementMeta(, 780df7fd-f32f-4aba-9a1c-c0e4bccb6235, 4, Finished, Available, Finished, False)

# Setting Up Parameters

_This block sets all parameters such as : ADLS Gen 2 Storage Account, File System Container, Workspaces and Item Types to include (blank for all), Service Principal details. It will also construct the Export root directory path for the Export Process_

- Added support for both Dataflow Gen2 and Dataflow Gen1
    - Dataflow Gen1 : Item_Type : DataflowGen1
    - Dataflow Gen2 : Item_Type : Dataflow

In [4]:
# ===== ADLS target =====
STORAGE_ACCOUNT = "storagetest****"
FILE_SYSTEM = "fabric-backup"   # container name in ADLS Gen2

# ===== optional filters =====
WORKSPACE_INCLUDE = ["test_move_1" ]   
# e.g. ["Finance", "Sales"]
ITEM_TYPE_INCLUDE = []   

#ITEM_TYPE_INCLUDE = ["Dataflow", "DataflowGen1"]   
# e.g. ["Notebook", "Report", "SemanticModel", "Dataflow", "DataflowGen1"]

# Secret names in Key Vault
TENANT_ID_SECRET = "94**6"
CLIENT_ID_SECRET = "0b**df"
CLIENT_SECRET_SECRET = "P***ycvT"
# ===== export root =====
EXPORT_ROOT = f"fabric-export/{datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')}"
ABFSS_BASE = f"abfss://{FILE_SYSTEM}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{EXPORT_ROOT}"

print(ABFSS_BASE)

StatementMeta(, 780df7fd-f32f-4aba-9a1c-c0e4bccb6235, 6, Finished, Available, Finished, False)

abfss://fabric-backup@storagetest00004.dfs.core.windows.net/fabric-export/2026-05-12T07-02-20Z


# Azure Data Lake Gen 2 connection

_This block sets spark configuration for ADLS Gen 2 connection via Service Principal

In [5]:
tenant_id = TENANT_ID_SECRET
client_id = CLIENT_ID_SECRET
client_secret = CLIENT_SECRET_SECRET

account_fqdn = f"{STORAGE_ACCOUNT}.dfs.core.windows.net"

spark.conf.set(f"fs.azure.account.auth.type.{account_fqdn}", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{account_fqdn}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(f"fs.azure.account.oauth2.client.id.{account_fqdn}", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{account_fqdn}", client_secret)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{account_fqdn}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

print("ABFSS OAuth configuration applied.")

StatementMeta(, 780df7fd-f32f-4aba-9a1c-c0e4bccb6235, 7, Finished, Available, Finished, False)

ABFSS OAuth configuration applied.


# Helper Functions

_This block contains Helper functions which are used in Export process_

In [6]:
from sempy.fabric import FabricRestClient


def sanitize_name(value: str) -> str:
    value = str(value or "").strip()
    value = re.sub(r'[<>:"/\\\\|?*]+', "_", value)
    value = re.sub(r"\s+", " ", value)
    return value[:180] if value else "unknown"


def ensure_folder(path: str):
    # Creates folder if it does not exist
    if not notebookutils.fs.exists(path):
        notebookutils.fs.mkdirs(path)


def write_text(path: str, text: str, overwrite: bool = True):
    parent = path.rsplit("/", 1)[0]
    ensure_folder(parent)
    notebookutils.fs.put(path, text, overwrite)


def write_bytes(path: str, payload: bytes, overwrite: bool = True):
    # notebookutils.fs.put writes text, so for binary files we go through Spark
    parent = path.rsplit("/", 1)[0]
    ensure_folder(parent)

    tmp_df = spark.createDataFrame([(bytearray(payload),)], ["content"])
    tmp_df.write.mode("overwrite" if overwrite else "errorifexists").format("binaryFile")
    raise NotImplementedError("Use write_binary_file() below instead")


def write_binary_file(path: str, payload: bytes):
    # Reliable binary write through Spark parallelize + saveAsTextFile is not ideal.
    # For most Fabric item-definition files, UTF-8 text/json is enough.
    # This function writes binary as base64 sidecar when binary is encountered.
    b64_path = path + ".base64.txt"
    write_text(b64_path, base64.b64encode(payload).decode("utf-8"), overwrite=True)
    return b64_path


def write_payload_part(base_folder: str, part: dict):
    rel_path = part.get("path", "unknown.bin")
    payload = part.get("payload")
    payload_type = part.get("payloadType")

    target_path = f"{base_folder}/{rel_path}"

    if payload_type == "InlineBase64" and payload is not None:
        raw = base64.b64decode(payload)

        # Try UTF-8 first because many item-definition files are json, tmdl, py, sql, etc.
        try:
            text = raw.decode("utf-8")
            write_text(target_path, text, overwrite=True)
            return target_path
        except UnicodeDecodeError:
            # Fallback: persist as base64 sidecar
            return write_binary_file(target_path, raw)

    # Unknown payload style: preserve metadata
    meta_path = target_path + ".metadata.json"
    write_text(meta_path, json.dumps(part, indent=2), overwrite=True)
    return meta_path

def get_pbi_access_token() -> str:
    """
    Gets a Power BI access token from the Fabric notebook runtime.
    """
    return notebookutils.credentials.getToken("pbi")


def pbi_headers() -> dict:
    token = get_pbi_access_token()
    return {
        "Authorization": f"Bearer {token}"
    }


def pbi_request_with_retry(method: str, url: str, **kwargs):
    """
    Basic retry wrapper for Power BI REST calls.
    """
    max_retries = 5
    default_retry_seconds = 15

    for attempt in range(1, max_retries + 1):
        resp = requests.request(method, url, **kwargs)

        if resp.status_code in (200, 202):
            return resp

        if resp.status_code == 429 or resp.status_code >= 500:
            retry_after = int(resp.headers.get("Retry-After", default_retry_seconds))
            print(f"Power BI API retryable response {resp.status_code}. Sleeping {retry_after}s (attempt {attempt}/{max_retries})")
            time.sleep(retry_after)
            continue

        raise RuntimeError(f"{resp.status_code} {resp.text}")

    raise RuntimeError(f"Power BI API failed after {max_retries} retries: {method} {url}")


def export_paginated_report_rdl(workspace_id: str, report_id: str) -> bytes:
    """
    Exports a paginated report as .rdl using Power BI REST API.
    Endpoint:
      GET https://api.powerbi.com/v1.0/myorg/groups/{groupId}/reports/{reportId}/Export
    For paginated reports, this returns the .rdl definition file.
    """
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/reports/{report_id}/Export"

    resp = pbi_request_with_retry(
        "GET",
        url,
        headers=pbi_headers(),
        stream=True
    )

    return resp.content

def write_binary_to_abfss(path: str, payload: bytes):
    """
    Writes binary content to ADLS Gen2 via Spark binaryFile datasource.
    """
    parent = path.rsplit("/", 1)[0]
    ensure_folder(parent)

    tmp_dir = f"{parent}/__tmp_binary_write__/{int(time.time() * 1000)}"
    local_tmp = f"/tmp/{int(time.time() * 1000)}.bin"

    # write local file first
    with open(local_tmp, "wb") as f:
        f.write(payload)

    # copy local file to ABFSS using notebookutils
    notebookutils.fs.cp(f"file:{local_tmp}", path)

def backup_semantic_model_rls(workspace_id: str, item_id: str, item_name: str, base_folder: str):
    """
    Backup semantic model role memberships and RLS permissions separately.
    Requires XMLA/TOM access and sufficient permissions on the semantic model.
    """

    # Roles + members
    roles_df = fabric.get_roles(
        dataset=item_id,
        workspace=workspace_id,
        include_members=True
    )

    # RLS permissions / filters
    rls_df = fabric.get_row_level_security_permissions(
        dataset=item_id,
        workspace=workspace_id
    )

    # Normalize nulls for JSON/CSV export
    roles_export_df = roles_df.fillna("")
    rls_export_df = rls_df.fillna("")

    # Write JSON
    write_text(
        f"{base_folder}/_roles_with_members.json",
        roles_export_df.to_json(orient="records", indent=2),
        overwrite=True
    )

    write_text(
        f"{base_folder}/_rls_permissions.json",
        rls_export_df.to_json(orient="records", indent=2),
        overwrite=True
    )

    # Write CSV
    write_text(
        f"{base_folder}/_roles_with_members.csv",
        roles_export_df.to_csv(index=False),
        overwrite=True
    )

    write_text(
        f"{base_folder}/_rls_permissions.csv",
        rls_export_df.to_csv(index=False),
        overwrite=True
    )

    # Write summary
    summary = {
        "workspace_id": workspace_id,
        "semantic_model_id": item_id,
        "semantic_model_name": item_name,
        "roles_row_count": int(len(roles_export_df)),
        "rls_permissions_row_count": int(len(rls_export_df)),
        "backup_utc": datetime.now(timezone.utc).isoformat()
    }

    write_text(
        f"{base_folder}/_rls_backup_summary.json",
        json.dumps(summary, indent=2),
        overwrite=True
    )

    return {
        "roles_row_count": int(len(roles_export_df)),
        "rls_permissions_row_count": int(len(rls_export_df))
    }

rest_client = FabricRestClient()

def get_item_definition_via_rest(workspace_id: str, item_id: str) -> dict:
    """
    Calls Fabric REST API directly because some runtimes don't expose
    sempy.fabric.get_item_definition().
    """
    path = f"/v1/workspaces/{workspace_id}/items/{item_id}/getDefinition"

    response = rest_client.post(path)

    if response.status_code == 200:
        return response.json()

    if response.status_code == 202:
        operation_id = response.headers.get("x-ms-operation-id")
        if not operation_id:
            raise RuntimeError("LRO started but x-ms-operation-id header was missing.")

        while True:
            op_resp = rest_client.get(f"/v1/operations/{operation_id}")
            op_json = op_resp.json()

            status = op_json.get("status")
            if status == "Succeeded":
                result_resp = rest_client.get(f"/v1/operations/{operation_id}/result")
                return result_resp.json()

            if status in ("Failed", "Canceled"):
                raise RuntimeError(f"Get definition failed for item {item_id}: {json.dumps(op_json, indent=2)}")

            retry_after = int(op_resp.headers.get("Retry-After", 10))
            time.sleep(retry_after)

    raise RuntimeError(f"{response.status_code} {response.text}")
from sempy.fabric import FabricRestClient
import json
import time

rest_client = FabricRestClient()

def fabric_request_with_retry(method: str, path: str, **kwargs):
    max_retries = 5
    default_retry_seconds = 10

    for attempt in range(1, max_retries + 1):
        response = getattr(rest_client, method)(path, **kwargs)

        if response.status_code in (200, 201, 202):
            return response

        if response.status_code == 429 or response.status_code >= 500:
            retry_after = int(response.headers.get("Retry-After", default_retry_seconds))
            print(f"Retryable Fabric API response {response.status_code}. Sleeping {retry_after}s (attempt {attempt}/{max_retries})")
            time.sleep(retry_after)
            continue

        raise RuntimeError(f"{response.status_code} {response.text}")

    raise RuntimeError(f"Fabric API failed after {max_retries} retries: {method.upper()} {path}")


def get_workspace_metadata(workspace_id: str) -> dict:
    """
    Backup workspace metadata/settings exposed by Fabric Core Get Workspace.
    """
    resp = fabric_request_with_retry("get", f"/v1/workspaces/{workspace_id}")
    return resp.json()


def list_workspace_role_assignments(workspace_id: str) -> list:
    """
    Backup workspace access list (users, groups, service principals, etc.)
    using Fabric Core role assignments API with pagination.
    """
    all_rows = []
    continuation_token = None

    while True:
        path = f"/v1/workspaces/{workspace_id}/roleAssignments"
        if continuation_token:
            path += f"?continuationToken={continuation_token}"

        resp = fabric_request_with_retry("get", path)
        payload = resp.json()

        all_rows.extend(payload.get("value", []))
        continuation_token = payload.get("continuationToken")

        if not continuation_token:
            break

    return all_rows


def normalize_workspace_role_assignments(role_assignments: list) -> pd.DataFrame:
    """
    Flatten Fabric workspace role assignments into a tabular structure.
    """
    rows = []

    for ra in role_assignments:
        principal = ra.get("principal", {}) or {}
        principal_type = principal.get("type", "")
        user_details = principal.get("userDetails", {}) or {}
        sp_details = principal.get("servicePrincipalDetails", {}) or {}
        spp_details = principal.get("servicePrincipalProfileDetails", {}) or {}
        parent_principal = spp_details.get("parentPrincipal", {}) or {}

        rows.append({
            "assignment_id": ra.get("id", ""),
            "workspace_role": ra.get("role", ""),
            "principal_type": principal_type,
            "principal_display_name": principal.get("displayName", ""),
            "principal_id": principal.get("id", ""),
            "user_principal_name": user_details.get("userPrincipalName", ""),
            "service_principal_app_id": sp_details.get("aadAppId", ""),
            "service_principal_profile_parent_type": parent_principal.get("type", ""),
            "service_principal_profile_parent_id": parent_principal.get("id", ""),
            "service_principal_profile_parent_display_name": parent_principal.get("displayName", "")
        })

    return pd.DataFrame(rows)


def backup_workspace_level_metadata(workspace_name: str, workspace_id: str):
    """
    Write workspace metadata and access files into the workspace folder
    before item iteration starts.
    """
    ws_folder = f"{sanitize_name(workspace_name)}__{workspace_id}"
    workspace_base_folder = f"{ABFSS_BASE}/{ws_folder}"

    # 1) Workspace metadata from Get Workspace
    workspace_metadata_status = "not_started"
    workspace_metadata_error = ""

    try:
        workspace_json = get_workspace_metadata(workspace_id)
        write_text(
            f"{workspace_base_folder}/_workspace.json",
            json.dumps(workspace_json, indent=2),
            overwrite=True
        )
        workspace_metadata_status = "exported"
    except Exception as ex:
        workspace_metadata_status = "error"
        workspace_metadata_error = str(ex)
        print(f"Warning: failed to back up workspace metadata for {workspace_name}: {ex}")

    # 2) Workspace access / role assignments
    workspace_access_status = "not_started"
    workspace_access_error = ""
    access_count = 0

    try:
        role_assignments = list_workspace_role_assignments(workspace_id)

        write_text(
            f"{workspace_base_folder}/_workspace_access_role_assignments.json",
            json.dumps(role_assignments, indent=2),
            overwrite=True
        )

        access_df = normalize_workspace_role_assignments(role_assignments).fillna("")

        write_text(
            f"{workspace_base_folder}/_workspace_access_role_assignments_flat.json",
            access_df.to_json(orient="records", indent=2),
            overwrite=True
        )

        write_text(
            f"{workspace_base_folder}/_workspace_access_role_assignments_flat.csv",
            dataframe_to_csv_text(access_df),
            overwrite=True
        )

        workspace_access_status = "exported"
        access_count = int(len(access_df))

    except Exception as ex:
        workspace_access_status = "error"
        workspace_access_error = str(ex)
        print(f"Warning: failed to back up workspace access for {workspace_name}: {ex}")

    # 3) Summary (clean — no OneLake fields anymore)
    workspace_summary = {
        "workspace_name": workspace_name,
        "workspace_id": workspace_id,
        "workspace_metadata_status": workspace_metadata_status,
        "workspace_metadata_error": workspace_metadata_error,
        "workspace_access_status": workspace_access_status,
        "workspace_access_error": workspace_access_error,
        "workspace_access_count": access_count,
        "backup_utc": datetime.now(timezone.utc).isoformat()
    }

    write_text(
        f"{workspace_base_folder}/_workspace_backup_summary.json",
        json.dumps(workspace_summary, indent=2),
        overwrite=True
    )

    return workspace_summary
def get_pbi_access_token() -> str:
    return notebookutils.credentials.getToken("pbi")


def pbi_headers() -> dict:
    return {
        "Authorization": f"Bearer {get_pbi_access_token()}"
    }

def get_dataset_users_in_group(workspace_id: str, dataset_id: str) -> list:
    """
    Returns direct semantic model permissions from Power BI REST API.
    """
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{dataset_id}/users"
    resp = pbi_request_with_retry("GET", url, headers=pbi_headers())
    payload = resp.json()
    return payload.get("value", [])

def dataframe_to_csv_text(df: pd.DataFrame) -> str:
    """
    Convert DataFrame to CSV text without using pandas.to_csv(),
    to avoid runtime-specific pandas formatter import issues.
    """
    safe_df = df.copy().fillna("")

    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_MINIMAL)

    writer.writerow([str(col) for col in safe_df.columns])

    for row in safe_df.itertuples(index=False, name=None):
        writer.writerow(["" if v is None else str(v) for v in row])

    return output.getvalue()


# -------------------------------------------------
# Dashboard inventory helpers
# -------------------------------------------------
def pbi_headers() -> dict:
    return {
        "Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"
    }

def pbi_request_with_retry(method: str, url: str, **kwargs):
    max_retries = 5
    default_retry_seconds = 15

    for attempt in range(1, max_retries + 1):
        resp = requests.request(method, url, **kwargs)

        if resp.status_code in (200, 201, 202):
            return resp

        if resp.status_code == 429 or resp.status_code >= 500:
            retry_after = int(resp.headers.get("Retry-After", default_retry_seconds))
            print(f"Power BI API retryable response {resp.status_code}. Sleeping {retry_after}s (attempt {attempt}/{max_retries})")
            time.sleep(retry_after)
            continue

        raise RuntimeError(f"{resp.status_code} {resp.text}")

    raise RuntimeError(f"Power BI API failed after {max_retries} retries: {method} {url}")

def get_dashboards_in_group(workspace_id: str) -> list:
    """
    Inventory classic Power BI dashboards in a workspace.
    """
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/dashboards"
    resp = pbi_request_with_retry("GET", url, headers=pbi_headers())
    payload = resp.json()
    return payload.get("value", [])

def get_dashboard_tiles_in_group(workspace_id: str, dashboard_id: str) -> list:
    """
    Inventory tiles for a classic Power BI dashboard.
    """
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/dashboards/{dashboard_id}/tiles"
    resp = pbi_request_with_retry("GET", url, headers=pbi_headers())
    payload = resp.json()
    return payload.get("value", [])

def backup_dashboard_inventory(workspace_name: str, workspace_id: str):
    """
    Write dashboard inventory files under the workspace folder.
    """
    ws_folder = f"{sanitize_name(workspace_name)}__{workspace_id}"
    workspace_base_folder = f"{ABFSS_BASE}/{ws_folder}"

    dashboard_status = "not_started"
    dashboard_error = ""
    dashboard_count = 0
    tile_count = 0

    try:
        dashboards = get_dashboards_in_group(workspace_id)

        write_text(
            f"{workspace_base_folder}/_dashboards_inventory.json",
            json.dumps(dashboards, indent=2),
            overwrite=True
        )

        dashboards_df = pd.DataFrame(dashboards).fillna("")
        if len(dashboards_df) > 0:
            write_text(
                f"{workspace_base_folder}/_dashboards_inventory.csv",
                dataframe_to_csv_text(dashboards_df),
                overwrite=True
            )

        dashboard_tiles = []
        for d in dashboards:
            dashboard_id = d.get("id", "")
            dashboard_name = d.get("displayName", "")
            if not dashboard_id:
                continue

            try:
                tiles = get_dashboard_tiles_in_group(workspace_id, dashboard_id)
                for t in tiles:
                    t["_dashboard_id"] = dashboard_id
                    t["_dashboard_name"] = dashboard_name
                dashboard_tiles.extend(tiles)
            except Exception as tile_ex:
                print(f"Warning: failed to inventory tiles for dashboard {dashboard_name}: {tile_ex}")

        write_text(
            f"{workspace_base_folder}/_dashboard_tiles_inventory.json",
            json.dumps(dashboard_tiles, indent=2),
            overwrite=True
        )

        tiles_df = pd.DataFrame(dashboard_tiles).fillna("")
        if len(tiles_df) > 0:
            write_text(
                f"{workspace_base_folder}/_dashboard_tiles_inventory.csv",
                dataframe_to_csv_text(tiles_df),
                overwrite=True
            )

        dashboard_status = "exported"
        dashboard_count = int(len(dashboards))
        tile_count = int(len(dashboard_tiles))

    except Exception as ex:
        dashboard_status = "error"
        dashboard_error = str(ex)
        print(f"Warning: failed to back up dashboard inventory for {workspace_name}: {ex}")

    return {
        "dashboard_inventory_status": dashboard_status,
        "dashboard_inventory_error": dashboard_error,
        "dashboard_count": dashboard_count,
        "dashboard_tile_count": tile_count
    }

# -------------------------------------------------
# Dataflow Gen1 helpers
# -------------------------------------------------

def list_dataflows_in_workspace(workspace_id: str):
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/dataflows"
    resp = pbi_request_with_retry("GET", url, headers=pbi_headers())
    return resp.json().get("value", [])

def export_dataflow_gen1_definition(workspace_id: str, dataflow_id: str) -> str:
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/dataflows/{dataflow_id}"
    resp = pbi_request_with_retry("GET", url, headers=pbi_headers())
    # API returns the exported JSON definition
    return resp.text

def get_dataflow_gen1_datasources(workspace_id: str, dataflow_id: str) -> dict:
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/dataflows/{dataflow_id}/datasources"
    resp = pbi_request_with_retry("GET", url, headers=pbi_headers())
    return resp.json()

def get_dataflow_gen1_upstream(workspace_id: str, dataflow_id: str) -> dict:
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/dataflows/{dataflow_id}/upstreamDataflows"
    resp = pbi_request_with_retry("GET", url, headers=pbi_headers())
    return resp.json()

def get_dataflow_gen1_transactions(workspace_id: str, dataflow_id: str) -> dict:
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/dataflows/{dataflow_id}/transactions"
    resp = pbi_request_with_retry("GET", url, headers=pbi_headers())
    return resp.json()

def update_dataflow_gen1_properties(workspace_id: str, dataflow_id: str, props: dict):
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/dataflows/{dataflow_id}"
    body = {
        k: v for k, v in props.items()
        if k in ["name", "description", "allowNativeQueries", "computeEngineBehavior"]
    }
    return pbi_request_with_retry("PATCH", url, headers={**pbi_headers(), "Content-Type": "application/json"}, json=body)

def update_dataflow_gen1_refresh_schedule(workspace_id: str, dataflow_id: str, schedule: dict):
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/dataflows/{dataflow_id}/refreshSchedule"
    body = {"value": schedule}
    return pbi_request_with_retry("PATCH", url, headers={**pbi_headers(), "Content-Type": "application/json"}, json=body)


StatementMeta(, 780df7fd-f32f-4aba-9a1c-c0e4bccb6235, 8, Finished, Available, Finished, False)

# CREATE WORKSPACE LIST

This block includes EXLUSION list for Workspaces and loops each one to discover ITEMS inside. It puts all Workspace and Items in dataframes.

In [9]:
def list_workspace_items_combined(workspace_id: str):
    combined = []

    # 1) Fabric items
    fabric_df = fabric.list_items(workspace=workspace_id)
    display(fabric_df)
    # Normalize column names once
    cols_lower = {str(c).strip().lower(): c for c in fabric_df.columns}

    id_col = cols_lower.get("id")
    name_col = (
        cols_lower.get("display name")
        or cols_lower.get("displayname")
        or cols_lower.get("name")
    )
    type_col = cols_lower.get("type")

    if id_col is None or name_col is None or type_col is None:
        raise Exception(f"Unexpected columns from fabric.list_items: {list(fabric_df.columns)}")

    for idx in fabric_df.index:
        combined.append({
            "Id": fabric_df.at[idx, id_col],
            "Name": fabric_df.at[idx, name_col],
            "Type": fabric_df.at[idx, type_col],
            "Source": "fabric"
        })

    # 2) Power BI Dataflow Gen1 items
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/dataflows"
    resp = pbi_request_with_retry("GET", url, headers=pbi_headers())
    dataflows = resp.json().get("value", [])
    display(dataflows)

    for df in dataflows:
        combined.append({
            "Id": df.get("objectId") or df.get("id"),
            "Name": df.get("name"),
            "Type": "DataflowGen1",
            "Generation": "Gen1",
            "Source": "powerbi-dataflows"
        })

    return combined

# Define excluded workspace names
WORKSPACE_EXCLUDE = [
    "Admin monitoring",
    "Microsoft Fabric Capacity Metrics",
    "Microsoft Fabric Chargeback Reporting"
]

workspaces_df = fabric.list_workspaces().copy()

# Apply include filter (if any)
if WORKSPACE_INCLUDE:
    workspaces_df = workspaces_df[workspaces_df["Name"].isin(WORKSPACE_INCLUDE)]

# Apply exclude filter
workspaces_df = workspaces_df[~workspaces_df["Name"].isin(WORKSPACE_EXCLUDE)]

display(workspaces_df[["Name", "Id", "Type"]].sort_values("Name"))

all_items = []

for _, ws in workspaces_df.iterrows():
    ws_name = ws["Name"]
    ws_id = ws["Id"]

    try:
        items = list_workspace_items_combined(workspace_id=ws_id)

        for item in items:
            item["WorkspaceName"] = ws_name
            item["WorkspaceId"] = ws_id
            all_items.append(item)

        print(f"Listed items for workspace: {ws_name}")

    except Exception as ex:
        print(f"Failed to list items for workspace {ws_name}: {ex}")

items_df = pd.DataFrame(all_items) if all_items else pd.DataFrame()

if not items_df.empty and ITEM_TYPE_INCLUDE:
    items_df = items_df[items_df["Type"].isin(ITEM_TYPE_INCLUDE)]

display(
  items_df[["WorkspaceName", "Name", "Type", "Id"]]
    .sort_values(["WorkspaceName", "Type", "Name"]))

print(f"Total items discovered: {len(items_df)}")

StatementMeta(, 780df7fd-f32f-4aba-9a1c-c0e4bccb6235, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d0b2ee41-e831-4f05-af06-25153ae26e19)

SynapseWidget(Synapse.DataFrame, 0cfdc492-4742-4537-80b6-8a592e8a8931)

Failed to list items for workspace test_move_1: 404 


KeyError: "None of [Index(['WorkspaceName', 'Name', 'Type', 'Id'], dtype='object')] are in the [columns]"

# EXPORT ITEMS

_This block loops around all items in each workspace and scripts out eacb item.
For Semantic Models, it also generates JSON and CSV files for RLS permissions and assignments. Roles are kept in TMDL files. For Paginated Reports, it generates RDL files._

In [ ]:
import io
import csv
import traceback
import requests

# -------------------------------------------------
# Workspace-level backup first
# -------------------------------------------------
workspace_backup_rows = []

workspace_scope_df = (
    items_df[["WorkspaceName", "WorkspaceId"]]
    .drop_duplicates()
    .sort_values(["WorkspaceName", "WorkspaceId"])
)

for _, ws in workspace_scope_df.iterrows():
    workspace_name = ws["WorkspaceName"]
    workspace_id = ws["WorkspaceId"]

    print(f"Backing up workspace-level metadata: {workspace_name}")

    try:
        workspace_summary = backup_workspace_level_metadata(
            workspace_name=workspace_name,
            workspace_id=workspace_id
        )

        # Add classic dashboard inventory
        dashboard_summary = backup_dashboard_inventory(
            workspace_name=workspace_name,
            workspace_id=workspace_id
        )
        workspace_summary.update(dashboard_summary)

        write_text(
            f"{ABFSS_BASE}/{sanitize_name(workspace_name)}__{workspace_id}/_workspace_backup_summary.json",
            json.dumps(workspace_summary, indent=2),
            overwrite=True
        )

        workspace_backup_rows.append(workspace_summary)

    except Exception as ws_ex:
        print(f"Warning: failed workspace-level backup for {workspace_name}")
        print(traceback.format_exc())

        workspace_backup_rows.append({
            "workspace_name": workspace_name,
            "workspace_id": workspace_id,
            "workspace_metadata_status": "error",
            "workspace_metadata_error": str(ws_ex),
            "workspace_access_status": "error",
            "workspace_access_error": str(ws_ex),
            "workspace_access_count": 0,
            "dashboard_inventory_status": "error",
            "dashboard_inventory_error": str(ws_ex),
            "dashboard_count": 0,
            "dashboard_tile_count": 0,
            "backup_utc": datetime.now(timezone.utc).isoformat()
        })

workspace_backup_df = pd.DataFrame(workspace_backup_rows)

write_text(
    f"{ABFSS_BASE}/_workspace_level_backup_manifest.json",
    workspace_backup_df.to_json(orient="records", indent=2),
    overwrite=True
)

# -------------------------------------------------
# Item-level backup
# -------------------------------------------------
manifest_rows = []

success_count = 0
skip_count = 0
error_count = 0

# Classic Power BI dashboards do not support Get Item Definition
UNSUPPORTED_DEFINITION_TYPES = {"Dashboard"}

for _, row in items_df.iterrows():
    workspace_name = row["WorkspaceName"]
    workspace_id = row["WorkspaceId"]
    item_name = row["Name"]
    item_id = row["Id"]
    item_type = row["Type"]

    ws_folder = f"{sanitize_name(workspace_name)}__{workspace_id}"
    item_folder = f"{sanitize_name(item_type)}/{sanitize_name(item_name)}__{item_id}"
    base_folder = f"{ABFSS_BASE}/{ws_folder}/{item_folder}"

    print(f"Exporting: {workspace_name} | {item_type} | {item_name} | {item_type}")

    rls_backup_status = ""
    rls_roles_count = 0
    rls_permissions_count = 0
    dataset_permissions_status = ""
    dataset_permissions_count = 0
    try:
        # ---- Skip unsupported item types up front ----
        if item_type in UNSUPPORTED_DEFINITION_TYPES:
            manifest_rows.append({
                "workspace_name": workspace_name,
                "workspace_id": workspace_id,
                "item_name": item_name,
                "item_id": item_id,
                "item_type": item_type,
                "status": "skipped_unsupported",
                "file_count": 0,
                "target_folder": base_folder,
                "rls_backup_status": "",
                "rls_roles_count": 0,
                "rls_permissions_count": 0,
                "dataset_permissions_status": "",
                "dataset_permissions_count": 0,
                "error": f"{item_type} items do not support Get Item Definition; inventoried at workspace level instead"
            })
            skip_count += 1
            print(f"Skipped unsupported item: {item_type} | {item_name}")
            continue

        # ---- Special handling for Paginated Reports ----
        if item_type == "PaginatedReport":
            rdl_bytes = export_paginated_report_rdl(workspace_id, item_id)

            rdl_path = f"{base_folder}/{sanitize_name(item_name)}.rdl"
            write_binary_to_abfss(rdl_path, rdl_bytes)

            metadata = {
                "workspace_name": workspace_name,
                "workspace_id": workspace_id,
                "item_name": item_name,
                "item_id": item_id,
                "item_type": item_type,
                "backup_format": "rdl",
                "backup_file": rdl_path,
                "exported_utc": datetime.now(timezone.utc).isoformat()
            }

            write_text(
                f"{base_folder}/_paginated_report_backup.json",
                json.dumps(metadata, indent=2),
                overwrite=True
            )

            manifest_rows.append({
                "workspace_name": workspace_name,
                "workspace_id": workspace_id,
                "item_name": item_name,
                "item_id": item_id,
                "item_type": item_type,
                "status": "exported",
                "file_count": 1,
                "target_folder": base_folder,
                "rls_backup_status": "",
                "rls_roles_count": 0,
                "rls_permissions_count": 0,
                "dataset_permissions_status": "",
                "dataset_permissions_count": 0,
                "error": ""
            })

            success_count += 1
            continue
         # ---- Special handling for Dataflow Gen1 ----

        if item_type == "DataflowGen1":
            
            model_json = export_dataflow_gen1_definition(workspace_id, item_id)
            datasources = get_dataflow_gen1_datasources(workspace_id, item_id)

            try:
                upstream = get_dataflow_gen1_upstream(workspace_id, item_id)
            except Exception:
                upstream = {"value": []}

            try:
                transactions = get_dataflow_gen1_transactions(workspace_id, item_id)
            except Exception:
                transactions = {"value": []}

            ## item_folder = f"{base_folder}/{sanitize_name(item_name)}"
            item_folder = f"{base_folder}"
            
            ensure_folder(item_folder)

            write_text(f"{item_folder}/model.json", model_json)
            write_text(f"{item_folder}/datasources.json", json.dumps(datasources, indent=2))
            write_text(f"{item_folder}/upstreamDataflows.json", json.dumps(upstream, indent=2))
            write_text(f"{item_folder}/transactions.json", json.dumps(transactions, indent=2))

            metadata = {
                "workspace_name": workspace_name,
                "workspace_id": workspace_id,
                "item_name": item_name,
                "item_id": item_id,
                "item_type": item_type,
                "backup_format": "dataflow_gen1_json",
                "exported_utc": datetime.now(timezone.utc).isoformat(),
                # keep only fields you know / can replay later
                "restore_notes": [
                    "Import model.json to create/restore the Gen1 dataflow",
                    "Reconfigure credentials/gateway after import",
                    "Reapply refresh schedule separately"
                ]
            }
            write_text(f"{item_folder}/metadata.json", json.dumps(metadata, indent=2))

            manifest_rows.append({
                "workspace_name": workspace_name,
                "workspace_id": workspace_id,
                "item_name": item_name,
                "item_id": item_id,
                "item_type": item_type,
                "status": "Exported",
                "file_count": 0,
                "target_folder": item_folder,
                "rls_backup_status": "",
                "rls_roles_count": 0,
                "rls_permissions_count": 0,
                "dataset_permissions_status": "",
                "dataset_permissions_count": 0,
                "error": ""
            })
            continue
        # ---- Standard handling for other supported Fabric items ----
        definition_json = get_item_definition_via_rest(
            workspace_id=workspace_id,
            item_id=item_id
        )

        if not isinstance(definition_json, dict):
            raise TypeError(
                f"get_item_definition_via_rest returned {type(definition_json)} instead of dict "
                f"for {item_type} | {item_name}"
            )

        write_text(
            f"{base_folder}/_definition_response.json",
            json.dumps(definition_json, indent=2),
            overwrite=True
        )

        definition = definition_json.get("definition", {})
        if not isinstance(definition, dict):
            raise TypeError(
                f"'definition' is {type(definition)} instead of dict for {item_type} | {item_name}"
            )

        parts = definition.get("parts", [])
        if parts is None:
            parts = []

        written_files = []
        for idx, part in enumerate(parts):
            try:
                written_path = write_payload_part(base_folder, part)
                written_files.append(written_path)
            except Exception as part_ex:
                raise RuntimeError(
                    f"Failed while writing definition part #{idx} for {item_type} | {item_name}. "
                    f"Part type: {type(part)}. Error: {part_ex}"
                ) from part_ex
        
            # ---- Extra backup only for Semantic Models ----
        if item_type == "SemanticModel":
            try:
                roles_df = fabric.get_roles(
                    dataset=item_id,
                    workspace=workspace_id,
                    include_members=True
                )

                rls_df = fabric.get_row_level_security_permissions(
                    dataset=item_id,
                    workspace=workspace_id
                )

                roles_export_df = roles_df.fillna("")
                rls_export_df = rls_df.fillna("")

                write_text(
                    f"{base_folder}/_roles_with_members.json",
                    roles_export_df.to_json(orient="records", indent=2),
                    overwrite=True
                )

                write_text(
                    f"{base_folder}/_rls_permissions.json",
                    rls_export_df.to_json(orient="records", indent=2),
                    overwrite=True
                )

                write_text(
                    f"{base_folder}/_roles_with_members.csv",
                    dataframe_to_csv_text(roles_export_df),
                    overwrite=True
                )

                write_text(
                    f"{base_folder}/_rls_permissions.csv",
                    dataframe_to_csv_text(rls_export_df),
                    overwrite=True
                )

                rls_summary = {
                    "workspace_name": workspace_name,
                    "workspace_id": workspace_id,
                    "semantic_model_name": item_name,
                    "semantic_model_id": item_id,
                    "roles_row_count": int(len(roles_export_df)),
                    "rls_permissions_row_count": int(len(rls_export_df)),
                    "backup_utc": datetime.now(timezone.utc).isoformat()
                }

                write_text(
                    f"{base_folder}/_rls_backup_summary.json",
                    json.dumps(rls_summary, indent=2),
                    overwrite=True
                )

                rls_backup_status = "exported"
                rls_roles_count = int(len(roles_export_df))
                rls_permissions_count = int(len(rls_export_df))

                print(f"Backed up RLS metadata for semantic model: {item_name}")

            except Exception as rls_ex:
                rls_backup_status = f"error: {str(rls_ex)}"
                print(f"Warning: failed to back up RLS metadata for semantic model {item_name}")
                print(traceback.format_exc())

            try:
                dataset_users = get_dataset_users_in_group(
                    workspace_id=workspace_id,
                    dataset_id=item_id
                )

                permissions_df = pd.DataFrame(dataset_users).fillna("")

                write_text(
                    f"{base_folder}/_dataset_permissions.json",
                    permissions_df.to_json(orient="records", indent=2),
                    overwrite=True
                )

                write_text(
                    f"{base_folder}/_dataset_permissions.csv",
                    dataframe_to_csv_text(permissions_df),
                    overwrite=True
                )

                dataset_permissions_status = "exported"
                dataset_permissions_count = int(len(permissions_df))

                print(f"Backed up direct semantic model permissions for: {item_name}")

            except Exception as perm_ex:
                dataset_permissions_status = f"error: {str(perm_ex)}"
                print(f"Warning: failed to back up direct semantic model permissions for semantic model {item_name}")
                print(traceback.format_exc())

        manifest_rows.append({
            "workspace_name": workspace_name,
            "workspace_id": workspace_id,
            "item_name": item_name,
            "item_id": item_id,
            "item_type": item_type,
            "status": "exported",
            "file_count": len(written_files),
            "target_folder": base_folder,
            "rls_backup_status": rls_backup_status,
            "rls_roles_count": rls_roles_count,
            "rls_permissions_count": rls_permissions_count,
            "dataset_permissions_status": dataset_permissions_status,
            "dataset_permissions_count": dataset_permissions_count,
            "error": ""
        })
        success_count += 1

    except Exception as ex:
        msg = str(ex)
        
        print("Full traceback:")
        print(traceback.format_exc())

        if "OperationNotSupportedForItem" in msg:
            manifest_rows.append({
                "workspace_name": workspace_name,
                "workspace_id": workspace_id,
                "item_name": item_name,
                "item_id": item_id,
                "item_type": item_type,
                "status": "skipped_unsupported",
                "file_count": 0,
                "target_folder": base_folder,
                "rls_backup_status": "",
                "rls_roles_count": 0,
                "rls_permissions_count": 0,
                "dataset_permissions_status": "",
                "dataset_permissions_count": 0,
                "error": msg
            })
            skip_count += 1
            print(f"Skipped unsupported item: {item_type} | {item_name}")

        elif "ItemHasProtectedLabel" in msg:
            manifest_rows.append({
                "workspace_name": workspace_name,
                "workspace_id": workspace_id,
                "item_name": item_name,
                "item_id": item_id,
                "item_type": item_type,
                "status": "skipped_protected_label",
                "file_count": 0,
                "target_folder": base_folder,
                "rls_backup_status": "",
                "rls_roles_count": 0,
                "rls_permissions_count": 0,
                "dataset_permissions_status": "",
                "dataset_permissions_count": 0,
                "error": msg
            })
            skip_count += 1
            print(f"Skipped protected-label item: {item_type} | {item_name}")

        else:
            manifest_rows.append({
                "workspace_name": workspace_name,
                "workspace_id": workspace_id,
                "item_name": item_name,
                "item_id": item_id,
                "item_type": item_type,
                "status": "error",
                "file_count": 0,
                "target_folder": base_folder,
                "rls_backup_status": "",
                "rls_roles_count": 0,
                "rls_permissions_count": 0,
                "dataset_permissions_status": "",
                "dataset_permissions_count": 0,
                "error": msg
            })
            error_count += 1
            print(f"ERROR: {item_name} -> {msg}")


# -------------------------------------------------
# Final manifests
# -------------------------------------------------
manifest_df = pd.DataFrame(manifest_rows)

write_text(
    f"{ABFSS_BASE}/_manifest.json",
    manifest_df.to_json(orient="records", indent=2),
    overwrite=True
)

write_text(
    f"{ABFSS_BASE}/_summary.json",
    json.dumps({
        "success_count": int(success_count),
        "skip_count": int(skip_count),
        "error_count": int(error_count),
        "backup_utc": datetime.now(timezone.utc).isoformat()
    }, indent=2),
    overwrite=True
)

display(workspace_backup_df)
display(manifest_df)

print(f"Success={success_count}, Skipped={skip_count}, Errors={error_count}")